# AI Bootcamp — Section 5, Practical 3: A2A (Local Protocol Lab)

**No AWS, no paid API, no flight bookings.** This is a **minimal learning implementation** of the A2A v0.3 **Agent Card + JSON-RPC `message/send`** contract. It is not the full `a2a-sdk` server (does not implement streaming, task persistence or authentication). The two specialty services return **fictional sample data**, while your Ollama Qwen3 produces a synthesis.

**Run code cells 1 → 8 in order.** This notebook uses Python standard-library threads instead of Windows Jupyter subprocesses (avoids `fileno` errors). All HTTP servers bind only to `127.0.0.1`.

## Cell 1 — Verify Python and local Ollama

In [ ]:
import sys, json, uuid, threading, atexit, time
from datetime import datetime
from pathlib import Path
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.request import build_opener, ProxyHandler, Request
from urllib.error import URLError, HTTPError

MODEL = "qwen3:1.7b"
OPENER = build_opener(ProxyHandler({}))  # Do not route localhost via corporate proxies

def fetch_json(url, payload=None, timeout=15):
    data = json.dumps(payload, ensure_ascii=False).encode('utf-8') if payload is not None else None
    req = Request(url, data=data, headers={'Content-Type':'application/json'} if data else {}, method='POST' if data else 'GET')
    with OPENER.open(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode('utf-8'))

tags = fetch_json('http://127.0.0.1:11434/api/tags')
models = [m.get('name','') for m in tags.get('models', [])]
assert any(m == MODEL for m in models), f'{MODEL} not found: {models}'
print('PASS: Ollama is running, model available:', MODEL)
print('Python:',sys.version.split()[0])

## Cell 2 — Fictional Weather & Flight Cases (not live data)

In [ ]:
SAMPLE = {
  'route': 'Ahmedabad (AMD) to Mumbai (BOM)',
  'weather': {'city':'Mumbai', 'temperature_c':28, 'conditions':'partly cloudy', 'rain_chance_pct':20},
  'flight': {'route':'Ahmedabad (AMD) to Mumbai (BOM)', 'flight':'DEMO-101', 'fare_inr':4500, 'available_seats':7}
}
assert SAMPLE['flight']['fare_inr'] > 0
print('PASS: Fictional training data loaded (NOT real weather or flight availability)')

## Cell 3 — Create two A2A-compatible local agent endpoints

In [ ]:
# A deliberately small subset of A2A 0.3: Agent Card + message/send.
def make_agent_handler(role):
    class AgentHandler(BaseHTTPRequestHandler):
        agent_role = role
        protocol_version = 'HTTP/1.1'
        def log_message(self, fmt, *args):
            pass  # Keep Jupyter output clean.
        def send_json(self, body, status=200):
            raw = json.dumps(body, ensure_ascii=False).encode('utf-8')
            self.send_response(status)
            self.send_header('Content-Type', 'application/json; charset=utf-8')
            self.send_header('Content-Length', str(len(raw)))
            self.end_headers()
            self.wfile.write(raw)
        def do_GET(self):
            if self.path != '/.well-known/agent-card.json':
                return self.send_json({'error':'Not found'},404)
            role = self.agent_role
            port = self.server.server_port
            skill_name = 'Sample Weather Analysis' if role=='weather' else 'Sample Flight Search'
            card = {
              'protocolVersion': '0.3.0',
              'name': f'{role.title()} Demo Agent',
              'description': 'Local demonstration using fictional training data; not live services',
              'url': f'http://127.0.0.1:{port}/',
              'preferredTransport': 'JSONRPC',
              'version': '0.1.0',
              'capabilities': {'streaming': False, 'pushNotifications': False},
              'defaultInputModes': ['text/plain'],
              'defaultOutputModes': ['text/plain'],
              'skills': [{'id':f'{role}_sample','name':skill_name,'description':'Only sample data','tags':['training',role]}]
            }
            self.send_json(card)
        def do_POST(self):
            if self.path != '/':
                return self.send_json({'error':'Not found'},404)
            try:
                length = int(self.headers.get('Content-Length','0'))
                if length <= 0 or length > 100000:
                    raise ValueError('Invalid payload size')
                req = json.loads(self.rfile.read(length).decode('utf-8'))
                rpc_id = req.get('id')
                if req.get('jsonrpc')!='2.0' or req.get('method')!='message/send':
                    return self.send_json({'jsonrpc':'2.0','id':rpc_id,'error':{'code':-32601,'message':'Method not found'}})
                message = req.get('params',{}).get('message',{})
                if message.get('role') != 'user' or not message.get('parts'):
                    raise ValueError('Expected a user text message')
                question = ' '.join(p.get('text','') for p in message['parts'] if p.get('kind')=='text')
                if not question.strip():
                    raise ValueError('No text part')
                if self.agent_role == 'weather':
                    w = SAMPLE['weather']
                    answer = f"TRAINING DATA ONLY: {w['city']} sample weather: {w['temperature_c']} C, {w['conditions']}, rain chance {w['rain_chance_pct']}%."
                else:
                    f = SAMPLE['flight']
                    answer = f"TRAINING DATA ONLY: {f['route']}, sample flight {f['flight']}, sample fare INR {f['fare_inr']}, sample seats {f['available_seats']}. Not bookable."
                response = {'jsonrpc':'2.0', 'id':rpc_id, 'result': {
                    'kind':'message', 'messageId':str(uuid.uuid4()), 'role':'agent',
                    'parts':[{'kind':'text','text':answer}]
                }}
                return self.send_json(response)
            except (ValueError, KeyError, json.JSONDecodeError) as exc:
                return self.send_json({'jsonrpc':'2.0','id':None,'error':{'code':-32602,'message':str(exc)}})
    return AgentHandler
print('PASS: Two independent A2A message handlers defined')

## Cell 4 — Start both servers inside Jupyter (no Windows subprocess)

In [ ]:
# Restart-safe: shut down any previous local demo servers before making new ones.
if 'A2A_SERVERS' in globals():
    for old in A2A_SERVERS.values():
        try: old.shutdown(); old.server_close()
        except Exception: pass
A2A_SERVERS = {}
A2A_THREADS = {}
for role in ('weather', 'flights'):
    server = ThreadingHTTPServer(('127.0.0.1',0), make_agent_handler(role))
    th = threading.Thread(target=server.serve_forever,daemon=True)
    th.start()
    A2A_SERVERS[role]=server
    A2A_THREADS[role]=th
    print(role, 'server on', f'http://127.0.0.1:{server.server_port}/')
assert all(t.is_alive() for t in A2A_THREADS.values())
print('PASS: 2 local A2A HTTP agent servers running')

## Cell 5 — Agent Card discovery (GET)

In [ ]:
CARDS = {}
for role, server in A2A_SERVERS.items():
    url = f'http://127.0.0.1:{server.server_port}/.well-known/agent-card.json'
    card = fetch_json(url)
    assert card['protocolVersion']=='0.3.0'
    assert card['preferredTransport']=='JSONRPC'
    assert len(card['skills'])==1
    CARDS[role] = card
    print(role, ':', card['name'], '->', card['url'])
print('PASS: Both A2A Agent Cards discovered')

## Cell 6 — Real inter-agent JSON-RPC message/send calls

In [ ]:
A2A_EVENTS = []
def call_agent(role, question):
    card = CARDS[role]
    req = {
        'jsonrpc':'2.0','id':str(uuid.uuid4()),'method':'message/send',
        'params':{'message':{'kind':'message','role':'user','messageId':str(uuid.uuid4()),
                             'parts':[{'kind':'text','text':question}]}}
    }
    response = fetch_json(card['url'],req)
    if 'error' in response: raise RuntimeError(response['error'])
    assert response['jsonrpc']=='2.0' and response['id']==req['id']
    result = response['result']
    assert result['kind']=='message' and result['role']=='agent'
    content=' '.join(p['text'] for p in result['parts'] if p['kind']=='text')
    A2A_EVENTS.append({'agent':role,'method':'message/send','request':question,'response':content})
    return content

weather_result = call_agent('weather','Describe sample Mumbai weather.')
flight_result = call_agent('flights','Describe a fictional Ahmedabad to Mumbai flight.')
print('Weather agent:', weather_result)
print('Flights agent:', flight_result)
assert 'TRAINING DATA ONLY' in weather_result and 'TRAINING DATA ONLY' in flight_result
print('PASS: 2 real HTTP JSON-RPC A2A calls completed')

## Cell 7 — Travel Orchestrator asks local Qwen3 for a combined report

In [ ]:
assert len(A2A_EVENTS) >= 2, 'Run Cell 6 first'
prompt = f"""You are a Hindi teacher explaining A2A. Use ONLY this fictional data.
Weather agent said: {weather_result}
Flights agent said: {flight_result}
Write 3-5 concise Hindi sentences about a sample travel plan. State clearly that weather, fare, flight, seats are FICTIONAL TRAINING DATA, not current facts or bookable tickets. Never invent prices or dates."""
body = {
 'model':MODEL, 'messages':[{'role':'user','content':prompt}],
 'stream':False,'think':False,
 'options':{'temperature':0.2,'num_predict':250}
}
ollama_data = fetch_json('http://127.0.0.1:11434/api/chat',body,timeout=180)
A2A_REPORT = (ollama_data.get('message') or {}).get('content','').strip()
assert A2A_REPORT, 'Ollama response was empty: check your running local model'
print('========== A2A LOCAL TRAVEL ORCHESTRATOR ==========')
print(A2A_REPORT)
print('PASS: Local Qwen3 synthesized both A2A agent responses')

## Cell 8 — Save the evidence to Markdown + JSON

In [ ]:
assert A2A_REPORT.strip() and len(A2A_EVENTS)>=2
md_path = Path('section5_a2a_local_report.md')
json_path = Path('section5_a2a_local_trace.json')
md_path.write_text(
    '# Section 5 — A2A Local Protocol Lab\n\n'
    '**Sample data only — no real weather, flight or booking.**\n\n'
    '## Weather agent\n' + weather_result + '\n\n'
    '## Flights agent\n' + flight_result + '\n\n'
    '## Qwen3 synthesis\n' + A2A_REPORT + '\n', encoding='utf-8'
)
json_path.write_text(json.dumps({
 'protocol':'A2A v0.3 JSON-RPC message/send educational subset',
 'transport':'localhost HTTP', 'model':MODEL,
 'agent_cards':CARDS,'events':A2A_EVENTS, 'summary':A2A_REPORT
},ensure_ascii=False,indent=2),encoding='utf-8')
assert md_path.exists() and json_path.exists()
print('Saved:',md_path.resolve())
print('Saved:',json_path.resolve())
print('PASS: Section 5 A2A local protocol report saved')

### What this proves / what it does not

You have used **live localhost HTTP connections**, two real **Agent Cards**, and two **A2A v0.3 JSON-RPC `message/send` calls**, followed by a **real local Ollama Qwen3** synthesis. This uses *fictional* training data and deliberately implements only a basic subset of A2A. It is **not** a complete SDK-based production A2A agent (no streaming, task store, auth, retries, A2A compliance testing or real travel APIs). For the official Bootcamp implementation, study `a2a-sdk>=0.3,<0.4` and its AgentExecutor / DefaultRequestHandler / A2AStarletteApplication workflow.

References: [Your Section 5 course](https://scaler-content.github.io/How-agents-talk/#a2a), [A2A 0.3 protocol specification](https://a2a-protocol.org/v0.3.0/specification/).